# SENTINEL · Notebook 5b · Public anchor on Polygon Amoy

Notebook 5 could not reach the blockchain because the old Polygon address it used no longer answers.
This short notebook:
1. updates the ledger model to try **four public Polygon Amoy connections** until one works,
2. checks your **test wallet** balance,
3. publishes your ledger's **Merkle roots** on the public test blockchain,
4. reads them back from the blockchain to prove they match.

It uses the same wallet Notebook 5 created (saved in Drive → SENTINEL → keys). Nothing else in Notebook 5 needs to be re-run.

Tap **☰ → Runtime → Run all** and allow Google Drive access.

## Step 1 · Load the tools and your saved files

In [1]:
!pip install -q web3
import json, os, sys, importlib, sqlite3
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/SENTINEL"
DATA_DIR, KEY_DIR = f"{BASE}/data", f"{BASE}/keys"
sys.path.insert(0, f"{BASE}/code/ledger")
needed = [f"{DATA_DIR}/anchors.json", f"{DATA_DIR}/ledger_export.json", f"{KEY_DIR}/amoy_testnet_key.txt"]
missing = [p.replace(BASE + "/", "") for p in needed if not os.path.exists(p)]
if missing:
    print("❌ Missing:", missing, "→ run Notebook 5 completely first.")
else:
    anchors = json.load(open(f"{DATA_DIR}/anchors.json"))
    export = json.load(open(f"{DATA_DIR}/ledger_export.json"))
    key = open(f"{KEY_DIR}/amoy_testnet_key.txt").read().strip()
    print(f"✅ Step 1 done: {len(anchors['roots'])} Merkle roots to publish, wallet key found")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.3/586.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.6/385.6 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.8/176.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.3 MB/s eta 0:00:00
Mounted at /content/drive
✅ Step 1 done: 11 Merkle roots to publish, wallet key found


## Step 2 · Update the ledger model file
Same model as before, now with a list of public Polygon Amoy connections (the first one is the address in Polygon's own documentation). It says **"Overwriting…"** when done.

In [2]:
%%writefile /content/drive/MyDrive/SENTINEL/code/ledger/ledger_model.py
# SENTINEL ledger/ · Tamper-evident ledger
#   hash_n = SHA-256(record_n ‖ hash_{n−1}) · Ed25519 signature per actor role
#   Merkle root per batch → public anchor (Polygon Amoy testnet) · verifier · no personal data
import hashlib
import json
import sqlite3
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey

GENESIS = "0" * 64


def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def sha256_hex(text):
    return hashlib.sha256(text.encode()).hexdigest()


def pseudonym(real_id, salt, prefix="PT"):
    # One-way pseudonym: the ledger never stores the real ID, name or phone number
    return f"{prefix}-{sha256_hex(salt + real_id)[:10]}"


def otp_hash(otp, trip_id):
    # The ledger stores only a hash of the handover OTP, never the OTP itself
    return sha256_hex(f"{otp}:{trip_id}")


class Signer:
    # One Ed25519 key per actor role: optimizer, control room, hospital, finance, commander, donor portal
    def __init__(self):
        self.keys = {}

    def _key(self, role):
        if role not in self.keys:
            self.keys[role] = Ed25519PrivateKey.generate()
        return self.keys[role]

    def sign(self, role, message):
        return self._key(role).sign(message.encode()).hex()

    def save(self, path):
        from cryptography.hazmat.primitives import serialization as ser
        raw = {r: k.private_bytes(ser.Encoding.Raw, ser.PrivateFormat.Raw, ser.NoEncryption()).hex()
               for r, k in self.keys.items()}
        with open(path, "w") as f:
            json.dump(raw, f)

    def load(self, path):
        with open(path) as f:
            for r, h in json.load(f).items():
                self.keys[r] = Ed25519PrivateKey.from_private_bytes(bytes.fromhex(h))
        return self

    def verify(self, role, message, signature_hex):
        try:
            self._key(role).public_key().verify(bytes.fromhex(signature_hex), message.encode())
            return True
        except Exception:
            return False


def merkle_root(hashes):
    level = list(hashes) or [GENESIS]
    while len(level) > 1:
        if len(level) % 2:
            level.append(level[-1])
        level = [sha256_hex(level[i] + level[i + 1]) for i in range(0, len(level), 2)]
    return level[0]


def merkle_proof(hashes, index):
    # The sibling hashes needed to recompute the root from one record
    proof, level, i = [], list(hashes), index
    while len(level) > 1:
        if len(level) % 2:
            level.append(level[-1])
        sib = i ^ 1
        proof.append({"hash": level[sib], "side": "left" if sib < i else "right"})
        level = [sha256_hex(level[k] + level[k + 1]) for k in range(0, len(level), 2)]
        i //= 2
    return proof


def verify_proof(leaf, proof, root):
    h = leaf
    for step in proof:
        h = sha256_hex(step["hash"] + h) if step["side"] == "left" else sha256_hex(h + step["hash"])
    return h == root


SCHEMA = """
CREATE TABLE IF NOT EXISTS ledger (
    seq INTEGER PRIMARY KEY, minute REAL, type TEXT, actor_role TEXT,
    payload TEXT, prev_hash TEXT, hash TEXT, signature TEXT);
CREATE TRIGGER IF NOT EXISTS ledger_no_update BEFORE UPDATE ON ledger
    BEGIN SELECT RAISE(ABORT, 'ledger is append-only'); END;
CREATE TRIGGER IF NOT EXISTS ledger_no_delete BEFORE DELETE ON ledger
    BEGIN SELECT RAISE(ABORT, 'ledger is append-only'); END;
CREATE TABLE IF NOT EXISTS batches (
    batch INTEGER PRIMARY KEY, first_seq INTEGER, last_seq INTEGER, merkle_root TEXT, anchor TEXT);
"""


class Ledger:
    def __init__(self, path, signer):
        self.db = sqlite3.connect(path)
        self.db.executescript(SCHEMA)
        self.signer = signer

    def last(self):
        row = self.db.execute("SELECT seq, hash FROM ledger ORDER BY seq DESC LIMIT 1").fetchone()
        return row if row else (0, GENESIS)

    def append(self, minute, rtype, actor_role, payload):
        seq, prev = self.last()
        body = {"seq": seq + 1, "minute": round(float(minute), 1), "type": rtype,
                "actor_role": actor_role, "payload": payload}
        h = sha256_hex(canonical(body) + prev)                 # hash_n = SHA-256(record_n ‖ hash_{n−1})
        sig = self.signer.sign(actor_role, h)
        self.db.execute("INSERT INTO ledger VALUES (?,?,?,?,?,?,?,?)",
                        (seq + 1, body["minute"], rtype, actor_role, canonical(payload), prev, h, sig))
        self.db.commit()
        return seq + 1, h

    def rows(self):
        cur = self.db.execute("SELECT seq, minute, type, actor_role, payload, prev_hash, hash, signature FROM ledger ORDER BY seq")
        return [{"seq": r[0], "minute": r[1], "type": r[2], "actor_role": r[3], "payload": json.loads(r[4]),
                 "prev_hash": r[5], "hash": r[6], "signature": r[7]} for r in cur]

    def verify_chain(self):
        # Recompute every hash, check every link and every signature
        prev = GENESIS
        for r in self.rows():
            body = {"seq": r["seq"], "minute": r["minute"], "type": r["type"],
                    "actor_role": r["actor_role"], "payload": r["payload"]}
            if r["prev_hash"] != prev:
                return False, r["seq"], "link to previous record broken"
            if sha256_hex(canonical(body) + prev) != r["hash"]:
                return False, r["seq"], "record content does not match its hash"
            if not self.signer.verify(r["actor_role"], r["hash"], r["signature"]):
                return False, r["seq"], "signature invalid"
            prev = r["hash"]
        return True, None, "all records, links and signatures valid"

    def seal_batch(self):
        # Merkle root over all records since the last batch
        last = self.db.execute("SELECT MAX(last_seq) FROM batches").fetchone()[0] or 0
        rows = [r for r in self.rows() if r["seq"] > last]
        if not rows:
            return None
        root = merkle_root([r["hash"] for r in rows])
        n = (self.db.execute("SELECT COUNT(*) FROM batches").fetchone()[0] or 0) + 1
        self.db.execute("INSERT INTO batches VALUES (?,?,?,?,?)", (n, rows[0]["seq"], rows[-1]["seq"], root, None))
        self.db.commit()
        return n, root

    def batches(self):
        cur = self.db.execute("SELECT batch, first_seq, last_seq, merkle_root, anchor FROM batches ORDER BY batch")
        return [{"batch": r[0], "first_seq": r[1], "last_seq": r[2], "merkle_root": r[3], "anchor": r[4]} for r in cur]

    def recompute_roots(self):
        rows = {r["seq"]: r["hash"] for r in self.rows()}
        return {b["batch"]: merkle_root([rows[s] for s in range(b["first_seq"], b["last_seq"] + 1) if s in rows])
                for b in self.batches()}

    def proof_for(self, seq):
        rows = {r["seq"]: r["hash"] for r in self.rows()}
        for b in self.batches():
            if b["first_seq"] <= seq <= b["last_seq"]:
                hashes = [rows[s] for s in range(b["first_seq"], b["last_seq"] + 1)]
                return rows[seq], merkle_proof(hashes, seq - b["first_seq"]), b
        return None


def role_view(role, record, viewer_hospital=None, viewer_donation=None):
    # Privacy controls: the same record looks different to each role
    p = record["payload"]
    if role == "Commander":
        return record
    if role == "Hospital":
        if p.get("hospital") == viewer_hospital:
            return record
        return {"seq": record["seq"], "type": record["type"], "hidden": "another hospital's record"}
    if role == "Donor":
        if p.get("donation") == viewer_donation:
            keep = {k: p[k] for k in ["donation", "category", "amount", "trip", "hospital", "delivered_verified"] if k in p}
            return {"seq": record["seq"], "type": record["type"], "payload": keep, "hash": record["hash"]}
        return {"seq": record["seq"], "type": record["type"], "hidden": "not your donation"}
    return {"seq": record["seq"], "type": record["type"], "minute": record["minute"], "hash": record["hash"][:16] + "…"}


# ---------- Public anchor on Polygon Amoy testnet ----------
AMOY_RPCS = [                                   # tried in order until one answers
    "https://polygon-amoy.drpc.org",            # listed in Polygon's own docs
    "https://polygon-amoy-bor-rpc.publicnode.com",
    "https://polygon-amoy.api.onfinality.io/public",
    "https://rpc-amoy.polygon.technology",
]
AMOY_CHAIN_ID = 80002
AMOY_EXPLORER = "https://amoy.polygonscan.com/tx/"


def connect_amoy(verbose=False):
    from web3 import Web3
    for url in AMOY_RPCS:
        try:
            w3 = Web3(Web3.HTTPProvider(url, request_kwargs={"timeout": 20}))
            if w3.is_connected() and w3.eth.chain_id == AMOY_CHAIN_ID:
                if verbose:
                    print(f"  ✅ {url}")
                return w3
            if verbose:
                print(f"  ❌ {url} (no answer)")
        except Exception as e:
            if verbose:
                print(f"  ❌ {url} ({str(e)[:60]})")
    return None


def anchor_on_amoy(private_key_hex, roots_hex):
    # Sends ONE transaction to yourself whose data field holds all batch Merkle roots
    from eth_account import Account
    w3 = connect_amoy()
    if w3 is None:
        return None, "could not reach the Polygon Amoy network on any public RPC"
    acct = Account.from_key(private_key_hex)
    if w3.eth.get_balance(acct.address) == 0:
        return None, "wallet has 0 test POL: get free test POL from a faucet first"
    base = w3.eth.get_block("latest").get("baseFeePerGas", w3.to_wei(30, "gwei"))
    tip = max(w3.eth.max_priority_fee, w3.to_wei(30, "gwei"))
    tx = {"to": acct.address, "value": 0, "data": "0x" + "".join(roots_hex), "chainId": AMOY_CHAIN_ID,
          "nonce": w3.eth.get_transaction_count(acct.address),
          "maxPriorityFeePerGas": tip, "maxFeePerGas": 2 * base + tip}
    tx["gas"] = int(w3.eth.estimate_gas(tx) * 1.2)
    signed = acct.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction).hex()
    if not tx_hash.startswith("0x"):
        tx_hash = "0x" + tx_hash
    w3.eth.wait_for_transaction_receipt(tx_hash, timeout=180)
    return tx_hash, "anchored"


def read_anchor_from_amoy(tx_hash):
    # Reads the roots back from the public blockchain (anyone can do this)
    w3 = connect_amoy()
    if w3 is None:
        raise ConnectionError("could not reach the Polygon Amoy network")
    data = w3.eth.get_transaction(tx_hash)["input"]
    data = data.hex() if hasattr(data, "hex") else str(data)
    data = data[2:] if data.startswith("0x") else data
    return [data[i:i + 64] for i in range(0, len(data), 64)]

Overwriting /content/drive/MyDrive/SENTINEL/code/ledger/ledger_model.py


## Step 3 · Test the connections

In [3]:
import ledger_model as lm
importlib.reload(lm)
print("Trying public Polygon Amoy connections:")
w3 = lm.connect_amoy(verbose=True)
print("✅ Step 3 done: connected" if w3 else "❌ None answered. Wait 5 minutes and run Step 3 again; public connections are sometimes busy.")

Trying public Polygon Amoy connections:
  ✅ https://polygon-amoy.drpc.org
✅ Step 3 done: connected


## Step 4 · Check your test wallet
If the balance is **0 POL**:
1. Long-press the address below → **Copy**.
2. Open **alchemy.com/faucets/polygon-amoy** in Chrome → paste → tap **Send**.
3. If Alchemy refuses, try **faucet.quicknode.com/polygon/amoy** (choose Polygon → Amoy, paste the address).
4. Wait 1–2 minutes, then run **Step 4 again**, then Steps 5 and 6.

Test POL has no real value. **Use this wallet only for testing.**

In [8]:
from eth_account import Account
address = Account.from_key(key).address
print("Test wallet address:", address)
if w3:
    pol = w3.eth.get_balance(address) / 1e18
    print(f"Balance: {pol:.4f} test POL")
    print("✅ Step 4 done: enough to anchor" if pol > 0 else "⏳ 0 POL: get test POL from the faucet (instructions above), then run Step 4 again")
    print("Wallet on the explorer: https://amoy.polygonscan.com/address/" + address)
else:
    print("❌ No connection: run Step 3 again first")

Test wallet address: 0xfe0Ac882e8306F97cf0C05866DBD794BfA4113c4
Balance: 0.0000 test POL
⏳ 0 POL: get test POL from the faucet (instructions above), then run Step 4 again
Wallet on the explorer: https://amoy.polygonscan.com/address/0xfe0Ac882e8306F97cf0C05866DBD794BfA4113c4


## Step 5 · Publish the Merkle roots on the blockchain
One transaction carries all the roots in its data field. This takes up to 1–2 minutes.
If it was already done, it reuses the existing transaction instead of paying again.

In [5]:
roots = anchors["roots"]
if anchors.get("tx"):
    tx = anchors["tx"]
    print("Already anchored earlier, reusing:", tx)
else:
    try:
        tx, msg = lm.anchor_on_amoy(key, roots)
    except Exception as e:
        tx, msg = None, f"failed: {str(e)[:150]}"
    if tx:
        anchors["tx"] = tx
        json.dump(anchors, open(f"{DATA_DIR}/anchors.json", "w"), indent=1)
        export["anchor"] = anchors
        json.dump(export, open(f"{DATA_DIR}/ledger_export.json", "w"), indent=1)
        db = sqlite3.connect(f"{DATA_DIR}/sentinel_ledger.db")
        db.execute("UPDATE batches SET anchor = ?", (tx,))
        db.commit()
    print("Result:", msg if not tx else "anchored")
if anchors.get("tx"):
    print("✅ Step 5 done · public proof:", lm.AMOY_EXPLORER + anchors["tx"])
    print("   Open this link: anyone in the world can see your roots and when they were published.")

Result: wallet has 0 test POL: get free test POL from a faucet first


## Step 6 · Read the roots back from the blockchain
This is what a judge, auditor or donor would do: fetch the transaction from the public network and compare it with the ledger.

In [6]:
if anchors.get("tx"):
    public = lm.read_anchor_from_amoy(anchors["tx"])
    ledger_roots = [b["merkle_root"] for b in export["batches"]]
    same = public[:len(ledger_roots)] == ledger_roots
    print(f"Roots on the blockchain: {len(public)} · roots in the ledger: {len(ledger_roots)}")
    print("✅ MATCH: the ledger is exactly what was published" if same else "🚨 MISMATCH: the ledger was changed after publishing")
    print("🎉 Notebook 5b complete. Send Claude a screenshot of Steps 5 and 6.")
else:
    print("⏳ Not anchored yet: finish Steps 4 and 5 first.")

⏳ Not anchored yet: finish Steps 4 and 5 first.


## Say this to a judge
> *"Every 15 minutes our ledger seals a Merkle root. All the roots are published in one transaction on the Polygon Amoy public blockchain. Here is the transaction on PolygonScan. If anyone, even an insider with every key, rewrites a single rupee, the recomputed roots no longer match what is public."*